# 🛡️ MTL-UEARDNet-v2: Multilingual Trimodal Hate Speech Detection


---

### Abstract
MTL-UEARDNet-v2 is a novel multilingual trimodal hate speech detection framework addressing three limitations of prior work: (1) English-only text encoders, (2) simple weighted modality fusion, and (3) binary-only classification. We replace HateBERT with XLM-RoBERTa-base for 100-language support, introduce a Transformer Cross-Modal Attention Fusion Block enabling bidirectional inter-modal information exchange, and adopt a three-head Multi-Task Learning framework jointly optimising binary detection, harm type classification, and targeted identity group identification. The system is evaluated on two independent held-out benchmarks: Multi3Hate (NAACL 2025, 5 languages) and HateMM (2023, audio-visual), with full XAI explainability and adversarial robustness testing.

### Dataset Overview
| Dataset | Year | Modalities | Languages | Annotations |
|---------|------|-----------|-----------|-------------|
| Multi3Hate | 2024 (NAACL 2025) | Text + Image | EN/DE/ES/HI/ZH | Binary + Target Category |
| UC Berkeley MHS | 2022 | Text | English | Binary + Harm Type + Target Group |
| HateMM | 2023 | Image + Audio | English | Binary + Target Group |
| CREMA-D | — | Audio | English | Emotion (audio encoder training) |

---


## ── SECTION 1: SETUP ───────────────────────────────────────────────────────

### What This Section Does
Installs all required libraries and sets global constants. Key additions over prior version:
- **`lime`**: adds LIME text explainability alongside SHAP
- **`matplotlib` / `networkx`**: used in Section 2 to generate architecture diagrams
- All permanent Kaggle dataset paths are declared here once and reused throughout


In [ ]:
# ── CELL 1: Install Dependencies ──────────────────────────────────────────────
!pip install transformers datasets librosa soundfile textattack shap lime funasr modelscope networkx -q
print("All packages installed ✓")


In [ ]:
# ── CELL 2: Imports & Global Constants ────────────────────────────────────────
import os, sys, json, warnings, random
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
import librosa
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import seaborn as sns
import shap

from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix, roc_curve, auc,
    precision_recall_fscore_support, matthews_corrcoef,
    average_precision_score, cohen_kappa_score,
    balanced_accuracy_score, f1_score as sk_f1,
    precision_recall_curve
)
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoTokenizer, AutoModel, CLIPModel, CLIPProcessor
from datasets import load_dataset

warnings.filterwarnings('ignore')
random.seed(42); np.random.seed(42); torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")

# ── Global constants ──────────────────────────────────────────────────────────
MAX_TEXT_LEN  = 128
HIDDEN_DIM    = 512
AUDIO_EMB_DIM = 768
W_BINARY = 0.50; W_TYPE = 0.20; W_TARGET = 0.20; W_RATIONALE = 0.10
MODEL_PATH = '/kaggle/working/mtl_ueardnet_v2_best.pth'

# ── Kaggle permanent dataset paths ───────────────────────────────────────────
MULTI3HATE_PATH    = '/kaggle/input/datasets/anshr8/multi3hate-2025/multi3hate_savedd'
MULTI3HATE_IMAGES  = '/kaggle/input/datasets/anshr8/multi3hate-2025/multi3hate_images'
CREMA_WAV_PATH     = '/kaggle/input/datasets/ejlok1/cremad/AudioWAV'
CREMA_EMB_PATH     = '/kaggle/input/models/anshr8/crema-model/keras/default/1/cremad_embeddings.npz'
HATEMM_PATH        = '/kaggle/input/datasets/anshr8/hatemm-processed-v2'
HATEMM_EMB_PATH    = '/kaggle/input/datasets/anshr8/hatemm-embeddings/hatemm_embeddings.npz'
HATEMM_FRAMES_PATH = f'{HATEMM_PATH}/hatemm_frames_v2'
print("Constants set ✓")


## ── SECTION 3: DATASET LOADING ─────────────────────────────────────────────

### Why Four Datasets?

No single dataset provides text + image + audio with both multilingual and multilabel support — an acknowledged open gap in 2025-2026 literature. The MTL framework bridges this: each head is supervised only by datasets that have the relevant labels (`-1` triggers loss masking for missing annotations).

| Dataset | Binary (H1) | Harm Type (H2) | Target Group (H3) | Modalities |
|---------|:-----------:|:--------------:|:-----------------:|-----------|
| Multi3Hate (2024) | ✓ | ✗ | ✓ | Text + Image |
| UC Berkeley MHS (2022) | ✓ | ✓ | ✓ | Text |
| HateMM (2023) | ✓ | ✗ | ✓ | Image + Audio |
| CREMA-D | ✓ | ✗ | ✗ | Audio |

### Multi3Hate (NAACL 2025)
- 1500 parallel memes in 5 languages — the most recent multilingual multimodal dataset
- Cultural annotations from USA, Germany, Mexico, India, China
- Only 300 samples per language — small but high-quality, culturally diverse

### UC Berkeley Measuring Hate Speech (2022)
- 50,000+ annotations from YouTube, Reddit, Twitter
- Continuous hate score + 10 survey dimensions + target identity columns
- Harm type derived via thresholding: hate_score≥0.5=Hate, high insult=Offensive, else Normal

### HateMM (2023)
- 1083 clips from BitChute with synchronized frame + emotion2vec audio embedding
- Target group from annotation CSV (Blacks/Muslims/Women/Others)

### CREMA-D
- 7442 acted emotional speech clips, 91 actors
- ANG/DIS/FEA → hate-adjacent (1), HAP/NEU/SAD → clean (0)


In [ ]:
# ── CELL 7: Load Multi3Hate (NAACL 2025) ────────────────────────────────────
print("Loading Multi3Hate...")
train_df_raw = pd.read_parquet('/kaggle/input/datasets/anshr8/multi3hate-2025/multi3hate_saved/train.parquet')
val_df_raw   = pd.read_parquet('/kaggle/input/datasets/anshr8/multi3hate-2025/multi3hate_saved/val.parquet')
test_df_raw  = pd.read_parquet('/kaggle/input/datasets/anshr8/multi3hate-2025/multi3hate_saved/test.parquet')

def fix_paths(df, base):
    df = df.copy()
    df['image_path'] = df['image_path'].apply(
        lambda p: os.path.join(base, os.path.basename(str(p))) if pd.notna(p) else None)
    return df

train_multi3hate = fix_paths(train_df_raw, MULTI3HATE_IMAGES)
val_multi3hate   = fix_paths(val_df_raw,   MULTI3HATE_IMAGES)
test_multi3hate  = fix_paths(test_df_raw,  MULTI3HATE_IMAGES)

CAT_MAP = {'Ethnicity':0,'Race':0,'Origin':0,'Religion':1,'Gender':2,'Sexuality':2,'Other':3}
for df in [train_multi3hate, val_multi3hate, test_multi3hate]:
    df['target_group'] = df.get('Category', pd.Series([None]*len(df))).map(
        lambda x: CAT_MAP.get(str(x).strip(), -1) if pd.notna(x) else -1)
    df['harm_type'] = -1

print(f"Multi3Hate — Train:{len(train_multi3hate)} Val:{len(val_multi3hate)} Test:{len(test_multi3hate)}")
print(f"Labels: {train_multi3hate['label'].value_counts().to_dict()}")
# Verify image
try:
    img = Image.open(train_multi3hate.iloc[0]['image_path'])
    print(f"Image OK: {img.size}")
except Exception as e:
    print(f"Image check: {e}")


In [ ]:
# ── CELL 8: Load UC Berkeley Measuring Hate Speech (2022) ────────────────────
print("Loading UC Berkeley MHS (2022)...")
ucb_ds  = load_dataset("ucberkeley-dlab/measuring-hate-speech", split="train")
ucb_raw = ucb_ds.to_pandas()
print(f"Raw rows: {len(ucb_raw):,}  Columns: {list(ucb_raw.columns[:15])}")

# Aggregate per comment
agg = {'text': 'first', 'hatespeech': 'mean'}
target_cols = [c for c in ucb_raw.columns if c.startswith('target_')]
score_cols  = [c for c in ['insult','humiliate','dehumanize','violence'] if c in ucb_raw.columns]
for c in target_cols + score_cols: agg[c] = 'max'

if 'comment_id' in ucb_raw.columns:
    ucb_df = ucb_raw.groupby('comment_id').agg(agg).reset_index()
else:
    ucb_df = ucb_raw.copy()

# Binary label
hs = ucb_df['hatespeech'].fillna(0) if 'hatespeech' in ucb_df.columns else pd.Series([0]*len(ucb_df))
if hs.max() > 1: hs = (hs - hs.min())/(hs.max() - hs.min() + 1e-8)
ucb_df['label'] = (hs >= 0.5).astype(int)

# Harm type
def harm(row):
    if row['label']==1: return 2
    ins = row.get('insult', 0) or 0
    return 1 if pd.notna(ins) and float(ins)>0.5 else 0
ucb_df['harm_type'] = ucb_df.apply(harm, axis=1)

# Target group
def tgroup(row):
    if row.get('target_race',0) or row.get('target_origin',0): return 0
    if row.get('target_religion',0):                            return 1
    if row.get('target_gender',0) or row.get('target_sexuality',0): return 2
    if row.get('target_age',0) or row.get('target_disability',0):   return 3
    return -1

ucb_df['target_group'] = ucb_df.apply(tgroup, axis=1) if target_cols else ucb_df['label'].apply(lambda x: 0 if x==1 else -1)

ucb_train, ucb_temp = train_test_split(ucb_df, test_size=0.30, random_state=42, stratify=ucb_df['label'])
ucb_val, ucb_test   = train_test_split(ucb_temp, test_size=0.667, random_state=42, stratify=ucb_temp['label'])
print(f"UCB splits — Train:{len(ucb_train)} Val:{len(ucb_val)} Test:{len(ucb_test)}")
print(f"Harm type dist: {ucb_df['harm_type'].value_counts().to_dict()}")


In [ ]:
print("Unique comments:", ucb_raw['comment_id'].nunique())
print("After aggregation:", len(ucb_df))

In [ ]:
# ── CELL 9: Load HateMM (2023) ───────────────────────────────────────────────
print("Loading HateMM (2023)...")
hatemm_csv = pd.read_csv('/kaggle/input/datasets/anshr8/hatemm-processed-v2/hatemm_processed_v2.csv')
hatemm_ann = pd.read_csv('/kaggle/input/datasets/anshr8/hatemm-annotations/HateMM_annotation.csv')
TM = {'Blacks':0,'Asians':0,'White':0,'Others':3,'Muslims':1,'Jews':1,'Christians':1,'Women':2,'LGBT':2,'lgbtq':2}
if 'target' in hatemm_ann.columns:
    hatemm_ann['target_group'] = hatemm_ann['target'].apply(lambda x: TM.get(str(x).strip(),3) if pd.notna(x) else -1)
    hatemm_df = hatemm_csv.merge(hatemm_ann[['video_file_name','target_group']].rename(columns={'video_file_name':'video_name'}),on='video_name',how='left')
else:
    hatemm_df = hatemm_csv.copy(); hatemm_df['target_group'] = hatemm_df['label'].apply(lambda x:0 if x==1 else -1)
hatemm_df['harm_type'] = -1

hatemm_emb_data  = np.load(HATEMM_EMB_PATH)
hatemm_embs      = hatemm_emb_data['embeddings']
hatemm_vid_ids   = hatemm_emb_data['video_ids']
hatemm_vid_map   = {v:i for i,v in enumerate(hatemm_vid_ids)}

hatemm_train, hatemm_temp = train_test_split(
    hatemm_df, test_size=0.30, random_state=42, stratify=hatemm_df['label'])
hatemm_val, hatemm_test = train_test_split(
    hatemm_temp, test_size=0.667, random_state=42, stratify=hatemm_temp['label'])
hatemm_train = hatemm_train.reset_index(drop=True)
hatemm_test  = hatemm_test.reset_index(drop=True)
print(f"HateMM — Train:{len(hatemm_train)} Test:{len(hatemm_test)}  EmbShape:{hatemm_embs.shape}")


## ── SECTION 4: AUDIO PREPARATION ────────────────────────────────────────────

### emotion2vec: Why Precomputed Embeddings?

**emotion2vec_plus_base** (IIC/FunASR, 2024) is a self-supervised audio representation model producing 768-dimensional utterance-level embeddings encoding prosodic cues (pitch, speech rate, energy), affective states, and spectral texture.


In [ ]:
# ── CELL 11: CREMA-D Audio Preparation ──────────────────────────────────────
HATE_EM  = {'ANG','DIS','FEA'}
CLEAN_EM = {'HAP','NEU','SAD'}
rows = []
for fname in os.listdir(CREMA_WAV_PATH):
    if not fname.endswith('.wav'): continue
    parts = fname.replace('.wav','').split('_')
    if len(parts)<3: continue
    em = parts[2].upper()
    if em in HATE_EM: lbl=1
    elif em in CLEAN_EM: lbl=0
    else: continue
    rows.append({'audio_path':CREMA_WAV_PATH+fname,'label':lbl,'emotion':em,'target_group':-1,'harm_type':-1})
audio_df = pd.DataFrame(rows)
print(f"CREMA-D: {len(audio_df):,}  (Hate:{(audio_df.label==1).sum()} Clean:{(audio_df.label==0).sum()})")
train_audio_df, val_audio_df = train_test_split(
    audio_df, test_size=0.30, stratify=audio_df.label, random_state=42)
print(f"Train/Val: {len(train_audio_df)}/{len(val_audio_df)}")


In [ ]:
# ── CELL 12: emotion2vec Embedding Extraction ────────────────────────────────
!pip install funasr modelscope -q
from funasr import AutoModel as FunASRModel

class _Quiet:
    def __enter__(self):
        self._dv=open(os.devnull,'w'); self._os=sys.stdout; self._es=sys.stderr
        sys.stdout=self._dv; sys.stderr=self._dv
        self._f1=os.dup(1); self._f2=os.dup(2)
        fd=os.open(os.devnull,os.O_WRONLY); os.dup2(fd,1); os.dup2(fd,2); os.close(fd)
        return self
    def __exit__(self,*a):
        sys.stdout=self._os; sys.stderr=self._es; self._dv.close()
        os.dup2(self._f1,1); os.dup2(self._f2,2); os.close(self._f1); os.close(self._f2)

if os.path.exists(CREMA_EMB_PATH):
    print("Loading precomputed CREMA-D embeddings...")
    d=np.load(CREMA_EMB_PATH); train_audio_embs=d['train_embeddings']; val_audio_embs=d['val_embeddings']
    AUDIO_EMB_DIM=train_audio_embs.shape[1]
    print(f"Train:{train_audio_embs.shape}  Val:{val_audio_embs.shape}  Dim:{AUDIO_EMB_DIM}")
else:
    print("Extracting emotion2vec embeddings...")
    with _Quiet(): e2v=FunASRModel(model='iic/emotion2vec_plus_base',hub='hf',disable_update=True)
    def extr(df, nm):
        out=[]
        for i,(_,r) in enumerate(df.iterrows()):
            if i%max(1,len(df)//10)==0: print(f"  {nm}: {int(i/len(df)*100)}%", flush=True)
            try:
                with _Quiet(): res=e2v.generate(r.audio_path,granularity='utterance',extract_embedding=True)
                out.append(np.array(res[0]['feats']).flatten())
            except: out.append(np.zeros(768,dtype=np.float32))
        return np.array(out,dtype=np.float32)
    train_audio_embs=extr(train_audio_df,'Train'); val_audio_embs=extr(val_audio_df,'Val')
    AUDIO_EMB_DIM=train_audio_embs.shape[1]
    np.savez(CREMA_EMB_PATH,train_embeddings=train_audio_embs,val_embeddings=val_audio_embs)
print("emotion2vec embeddings ready ✓")


## ── SECTION 5: ENCODER LOADING & FINE-TUNING ───────────────────────────────

### XLM-RoBERTa vs HateBERT — The Key Encoder Upgrade

HateBERT was trained on 1.5M English Reddit posts. Multi3Hate contains German, Spanish, Hindi, and Mandarin text — HateBERT produces random embeddings for these languages, making any multilingual result meaningless. XLM-RoBERTa-base solves this with consistent representations across 100 languages.

Kumar et al. (IEEE 2025) directly compared the two on multilingual hate speech: XLM-R achieved F1=92.6% vs HateBERT's 86.3%, a 6.3% absolute improvement.

### Partial Fine-tuning — Why Not Freeze Everything?

- **Fully frozen** (original approach): All encoder parameters locked. Fusion block only. Limited domain adaptation.
- **Fully fine-tuned**: All parameters updated. High overfitting risk, catastrophic forgetting, requires large dataset.
- **Partial unfreeze** (our approach, IEEE 2025): Last 3 XLM-R + last 2 CLIP layers + pooler are unfrozen. Preserves general knowledge, adapts to hate-specific features. Increases trainable params from ~0.6% to ~4%.


In [ ]:
# ── CELL 13: Load XLM-R + CLIP with Partial Fine-tuning ────────────────────
print("Loading XLM-RoBERTa-base...")
xlmr_tok   = AutoTokenizer.from_pretrained('xlm-roberta-base')
xlmr_model = AutoModel.from_pretrained('xlm-roberta-base')
for p in xlmr_model.parameters(): p.requires_grad=False
for l in xlmr_model.encoder.layer[-3:]:
    for p in l.parameters(): p.requires_grad=True
if hasattr(xlmr_model,'pooler') and xlmr_model.pooler:
    for p in xlmr_model.pooler.parameters(): p.requires_grad=True
xt = sum(p.numel() for p in xlmr_model.parameters() if p.requires_grad)
xn = sum(p.numel() for p in xlmr_model.parameters())
print(f"  XLM-R: {xt:,} trainable / {xn:,} total ({100*xt/xn:.1f}%)")

print("Loading CLIP-ViT-B/32...")
clip_proc  = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
for p in clip_model.parameters(): p.requires_grad=False
for l in clip_model.vision_model.encoder.layers[-2:]:
    for p in l.parameters(): p.requires_grad=True
ct = sum(p.numel() for p in clip_model.parameters() if p.requires_grad)
cn = sum(p.numel() for p in clip_model.parameters())
print(f"  CLIP : {ct:,} trainable / {cn:,} total ({100*ct/cn:.1f}%)")
print("Encoders ready ✓")


## ── SECTION 15: V3.2 — CORRECTED PIPELINE + SECOND-ROUND REVIEWER EXPERIMENTS ──

This section re-runs the study on a corrected pipeline. Every change below was found during the code audit for the second review round:

| # | Problem in V1/V2 code | Fix in V3 |
|---|---|---|
| 1 | KL rationale term was always zero (`tokens`/`rationale` were `None` for every row; HateXplain was never loaded) | Term removed; loss = 0.50·H1 + 0.20·H2 + 0.20·H3, which is what effectively trained before |
| 2 | Text stream never gated: HateMM/CREMA-D rows fed XLM-R's encoding of an empty string | Text flag = 1 only when a caption/comment exists; absent text is a zero vector |
| 3 | No attention masking over absent modality tokens | `src_key_padding_mask` + pooling over present tokens only (`tcaf_masked`) |
| 4 | HateMM *training* rows read CREMA-D embeddings (their `audio_idx` pointed into the CREMA block of the stacked array) | Embeddings are appended row-by-row, so each index is correct by construction |
| 5 | CREMA-D embeddings came from an npz with a different split (6,697/745) than the labels (5,209/2,233), so embeddings and labels were misaligned | Re-extracted and keyed by filename; actor-disjoint 70/30 split; linear-probe sanity check |
| 6 | `CREMA_WAV_PATH + fname` was missing a path separator | `os.path.join` |
| 7 | HateMM excluded from validation | HateMM val included in checkpoint selection |
| 8 | Multi3Hate split at row level: the 5 language versions of one meme (distinct image files, same `Meme ID`) could fall in both train and test | Audited by `Meme ID` (and file hash); split grouped by `Meme ID` if any overlap |
| 9 | HateMM target group: non-hate videos defaulted to "Other"; list-formatted (`['Jews']`), comma-separated and variant spellings (`Whites`, `LGBTQ`) also fell through to "Other" | Raw targets parsed in all formats; target group only for hateful videos; first-listed target for multi-target videos |
| 10 | UCB harm type: "Offensive" if *any* annotator gave insult > 0.5 (71% of comments) | "Offensive" if the *mean* annotator insult rating ≥ scale midpoint |

Every configuration is trained with 3 seeds, and each result reports bootstrap 95% CIs. Paired bootstrap tests compare configurations on identical test samples. Runs are cached in `/kaggle/working/v3/runs/`, so an interrupted session resumes where it stopped.

**Run order:** Run All. V3.2 writes to `/kaggle/working/v32`, so no earlier run is reused. Every V3 cell prints its own run time (⏱).

In [ ]:
import time as _time; _tc = _time.time()
# ── V3-0: Configuration ────────────────────────────────────────────────────────
import time, glob, hashlib, traceback, shutil
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, f1_score

V3_DIR = '/kaggle/working/v32'          # fresh directory: no earlier (pre-fix) run is ever reused
for sub in ['', 'runs', 'preds', 'ckpt', 'tables']:
    os.makedirs(os.path.join(V3_DIR, sub), exist_ok=True)

SEEDS        = [42, 43, 44]
EPOCHS_V3    = 6
PATIENCE_V3  = 2
BATCH_V3     = 32
LR_V3        = 2e-5
USE_AMP      = torch.cuda.is_available()   # fp16 autocast for speed; identical across all V3 runs
NUM_WORKERS  = min(4, os.cpu_count() or 2)
N_BOOT       = 1000
M3H_RESPLIT  = 'auto'      # 'auto' = re-split by image hash only if train/test overlap is found
FORCE_RETRAIN_V3 = False
SAVE_CKPT_FOR    = {'tcaf_masked'}   # checkpoints kept only for the main model (disk budget)
W_B3, W_T3, W_G3 = 0.50, 0.20, 0.20

AUDIT = {}
def save_audit():
    with open(f'{V3_DIR}/audit.json', 'w') as f:
        json.dump(AUDIT, f, indent=2, default=str)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

print(f"V3 config ready  |  seeds={SEEDS}  epochs={EPOCHS_V3}  AMP={USE_AMP}")
print(f"⏱ cell finished in {(_time.time()-_tc)/60:.1f} min")


In [ ]:
import time as _time; _tc = _time.time()
# ── V3-A: CREMA-D — rebuild, re-extract emotion2vec by filename, sanity-check ──
# Documents the V1/V2 misalignment first, then fixes it.

# (A1) Evidence for the old pipeline's misalignment (uses the objects from Cells 11/12)
try:
    _n_tr = min(len(train_audio_df), len(train_audio_embs))
    _n_va = min(len(val_audio_df), len(val_audio_embs))
    _probe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
    _probe.fit(train_audio_embs[:_n_tr], train_audio_df['label'].values[:_n_tr])
    AUDIT['old_crema_npz_shapes'] = {'train': list(train_audio_embs.shape), 'val': list(val_audio_embs.shape)}
    AUDIT['old_crema_df_sizes']   = {'train': len(train_audio_df), 'val': len(val_audio_df)}
    AUDIT['old_pipeline_crema_probe_auc'] = float(roc_auc_score(
        val_audio_df['label'].values[:_n_va], _probe.predict_proba(val_audio_embs[:_n_va])[:, 1]))
    print(f"[old pipeline] CREMA-D linear-probe val AUC with V1 alignment: {AUDIT['old_pipeline_crema_probe_auc']:.3f}  (≈0.5 ⇒ misaligned)")
except Exception as e:
    print("[old pipeline] probe skipped:", e)

# (A2) Rebuild CREMA-D table with correct paths and actor IDs
HATE_EM, CLEAN_EM = {'ANG','DIS','FEA'}, {'HAP','NEU','SAD'}
_rows = []
for fname in sorted(os.listdir(CREMA_WAV_PATH)):
    if not fname.endswith('.wav'): continue
    parts = fname[:-4].split('_')
    if len(parts) < 3: continue
    em = parts[2].upper()
    if em in HATE_EM:   lbl = 1
    elif em in CLEAN_EM: lbl = 0
    else: continue
    _rows.append({'fname': fname, 'audio_path': os.path.join(CREMA_WAV_PATH, fname),
                  'label': lbl, 'emotion': em, 'actor': parts[0]})
crema_df = pd.DataFrame(_rows)
print(f"CREMA-D files: {len(crema_df):,}  actors: {crema_df.actor.nunique()}  emotions: {crema_df.emotion.value_counts().to_dict()}")

# (A3) Extract (or load cached) embeddings keyed by filename
CREMA_V3_NPZ = f'{V3_DIR}/cremad_emotion2vec_by_fname.npz'
for _old in [f'/kaggle/working/{v}/cremad_emotion2vec_by_fname.npz' for v in ('v3', 'v31')] + glob.glob('/kaggle/input/**/cremad_emotion2vec_by_fname.npz', recursive=True):
    if not os.path.exists(CREMA_V3_NPZ) and os.path.exists(_old): shutil.copy(_old, CREMA_V3_NPZ)
if os.path.exists(CREMA_V3_NPZ):
    _d = np.load(CREMA_V3_NPZ, allow_pickle=True)
    emb_by_f = dict(zip(_d['fnames'].tolist(), _d['embs']))
    print(f"Loaded cached V3 CREMA-D embeddings: {len(emb_by_f):,}")
else:
    from funasr import AutoModel as FunASRModel
    with _Quiet(): e2v = FunASRModel(model='iic/emotion2vec_plus_base', hub='hf', disable_update=True)
    _fn, _em, _fail = [], [], []
    t0 = time.time()
    for i, r in enumerate(crema_df.itertuples(index=False)):
        if i % 500 == 0: print(f"  extracting {i:,}/{len(crema_df):,}  ({time.time()-t0:.0f}s)", flush=True)
        try:
            with _Quiet(): res = e2v.generate(r.audio_path, granularity='utterance', extract_embedding=True)
            v = np.asarray(res[0]['feats'], dtype=np.float32).flatten()
            if not np.isfinite(v).all() or np.abs(v).sum() == 0: raise ValueError('empty embedding')
            _fn.append(r.fname); _em.append(v)
        except Exception:
            _fail.append(r.fname)
    np.savez(CREMA_V3_NPZ, fnames=np.array(_fn), embs=np.stack(_em))
    emb_by_f = dict(zip(_fn, _em))
    AUDIT['crema_extraction_failures'] = len(_fail)
    print(f"Extracted {len(_fn):,}  failures: {len(_fail)}")
    del e2v; torch.cuda.empty_cache()

crema_df = crema_df[crema_df.fname.isin(emb_by_f)].reset_index(drop=True)
assert len(crema_df) > 0.99 * len(_rows), "More than 1% of CREMA-D clips failed extraction — check paths."
AUDIO_EMB_DIM = len(next(iter(emb_by_f.values())))

# (A4) Actor-disjoint 70/30 split (no speaker appears in both)
_gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
_tr_i, _va_i = next(_gss.split(crema_df, groups=crema_df.actor))
crema_train = crema_df.iloc[_tr_i].reset_index(drop=True)
crema_val   = crema_df.iloc[_va_i].reset_index(drop=True)
crema_train_embs = np.stack([emb_by_f[f] for f in crema_train.fname]).astype(np.float32)
crema_val_embs   = np.stack([emb_by_f[f] for f in crema_val.fname]).astype(np.float32)
assert set(crema_train.actor).isdisjoint(set(crema_val.actor))

# (A5) Sanity: a linear probe on correctly aligned embeddings must be far above chance
_probe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)).fit(crema_train_embs, crema_train.label)
AUDIT['v3_crema_probe_val_auc'] = float(roc_auc_score(crema_val.label, _probe.predict_proba(crema_val_embs)[:, 1]))
AUDIT['v3_crema_split'] = {'train': len(crema_train), 'val': len(crema_val),
                           'train_actors': crema_train.actor.nunique(), 'val_actors': crema_val.actor.nunique()}
print(f"[V3] CREMA-D linear-probe val AUC (actor-disjoint): {AUDIT['v3_crema_probe_val_auc']:.3f}")
assert AUDIT['v3_crema_probe_val_auc'] > 0.65, "CREMA-D embeddings still look misaligned — stop and inspect."

# (A6) HateMM audio embeddings: probe for information (no assertion)
_hm_idx = [hatemm_vid_map.get(v, -1) for v in hatemm_df['video_id'].values]
_ok = np.array([i >= 0 for i in _hm_idx])
AUDIT['hatemm_videos_with_audio_embedding'] = f"{_ok.sum()}/{len(_ok)}"
_X = hatemm_embs[[i for i in _hm_idx if i >= 0]]; _y = hatemm_df['label'].values[_ok]
AUDIT['hatemm_audio_probe_cv_auc'] = float(cross_val_score(
    make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)), _X, _y, cv=5, scoring='roc_auc').mean())
print(f"HateMM audio-only linear-probe 5-fold AUC: {AUDIT['hatemm_audio_probe_cv_auc']:.3f}  (videos with audio: {AUDIT['hatemm_videos_with_audio_embedding']})")

# (A7) Evidence for the old HateMM audio-index bug
try:
    _old_idx = [hatemm_vid_map.get(v, -1) for v in hatemm_train['video_id'].values]
    AUDIT['old_pipeline_hatemm_train_rows_reading_crema_block'] = int(sum(0 <= i < len(train_audio_embs) for i in _old_idx))
    print(f"[old pipeline] HateMM train rows whose audio index fell inside the CREMA-D block: "
          f"{AUDIT['old_pipeline_hatemm_train_rows_reading_crema_block']}/{len(_old_idx)}")
except Exception as e:
    print("[old pipeline] HateMM index check skipped:", e)
save_audit()
print(f"⏱ cell finished in {(_time.time()-_tc)/60:.1f} min")


In [ ]:
import time as _time; _tc = _time.time()
# ── V3-B: Multi3Hate leakage audit (file hash) + grouped re-split if needed ──
def _file_md5(p):
    try:
        with open(p, 'rb') as f: return hashlib.md5(f.read()).hexdigest()
    except Exception:
        return None

print("Multi3Hate columns:", list(train_multi3hate.columns))
for _df in (train_multi3hate, val_multi3hate, test_multi3hate):
    _df['img_md5'] = _df['image_path'].map(lambda p: _file_md5(p) if isinstance(p, str) else None)

_KEY = 'Meme ID' if 'Meme ID' in train_multi3hate.columns else 'img_md5'
_tr_md5, _va_md5 = set(train_multi3hate.img_md5.dropna()), set(val_multi3hate.img_md5.dropna())
AUDIT['m3h_missing_images'] = int(sum(d.img_md5.isna().sum() for d in (train_multi3hate, val_multi3hate, test_multi3hate)))
AUDIT['m3h_unique_image_files'] = int(pd.concat([train_multi3hate, val_multi3hate, test_multi3hate]).img_md5.nunique())
AUDIT['m3h_test_rows_same_image_file_in_train_or_val'] = f"{int(test_multi3hate.img_md5.isin(_tr_md5 | _va_md5).sum())}/{len(test_multi3hate)}"
_tr_id, _va_id = set(train_multi3hate[_KEY]), set(val_multi3hate[_KEY])
_te_leak = test_multi3hate[_KEY].isin(_tr_id | _va_id)
AUDIT['m3h_group_key'] = _KEY
AUDIT['m3h_unique_memes'] = int(pd.concat([train_multi3hate, val_multi3hate, test_multi3hate])[_KEY].nunique())
AUDIT['m3h_test_rows_same_meme_in_train_or_val'] = f"{int(_te_leak.sum())}/{len(test_multi3hate)}"
print(f"Unique image files: {AUDIT['m3h_unique_image_files']}   unique memes ({_KEY}): {AUDIT['m3h_unique_memes']}")
print(f"Test rows with the same image FILE in train/val: {AUDIT['m3h_test_rows_same_image_file_in_train_or_val']}")
print(f"Test rows with the same MEME (another language version) in train/val: {AUDIT['m3h_test_rows_same_meme_in_train_or_val']}")

_do_resplit = (M3H_RESPLIT is True) or (M3H_RESPLIT == 'auto' and _te_leak.any())
AUDIT['m3h_resplit_applied'] = bool(_do_resplit)
if _do_resplit:
    _all = pd.concat([train_multi3hate, val_multi3hate, test_multi3hate], ignore_index=True)
    _g = _all[_KEY].astype(str)
    _sg = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    _trv_i, _te_i = next(_sg.split(_all, _all.label, groups=_g))
    _trv = _all.iloc[_trv_i].reset_index(drop=True)
    _sg2 = StratifiedGroupKFold(n_splits=8, shuffle=True, random_state=42)
    _tr_i, _va_i = next(_sg2.split(_trv, _trv.label, groups=_g.iloc[_trv_i].values))
    M3H_TR, M3H_VA, M3H_TE = _trv.iloc[_tr_i], _trv.iloc[_va_i], _all.iloc[_te_i]
    print(f"Applied {_KEY}-grouped re-split: every language version of a meme stays in one split.")
else:
    M3H_TR, M3H_VA, M3H_TE = train_multi3hate, val_multi3hate, test_multi3hate
    print("No overlap found — original Multi3Hate split kept.")

M3H_TR, M3H_VA, M3H_TE = (d.reset_index(drop=True) for d in (M3H_TR, M3H_VA, M3H_TE))
assert set(M3H_TE[_KEY]).isdisjoint(set(M3H_TR[_KEY]) | set(M3H_VA[_KEY])) or not _do_resplit
AUDIT['m3h_split_sizes'] = {'train': len(M3H_TR), 'val': len(M3H_VA), 'test': len(M3H_TE)}
AUDIT['m3h_test_label_counts'] = M3H_TE['label'].value_counts().to_dict()
if 'Language' in M3H_TE.columns:
    AUDIT['m3h_test_language_counts'] = M3H_TE['Language'].value_counts().to_dict()
print("Split sizes:", AUDIT['m3h_split_sizes'], " test languages:", AUDIT.get('m3h_test_language_counts'))
save_audit()
print(f"⏱ cell finished in {(_time.time()-_tc)/60:.1f} min")


In [ ]:
import time as _time; _tc = _time.time()
# ── V3-C: Label-rule corrections + mapping transparency (for the manuscript's mapping table) ──
# (C1) HateMM target group: parse raw annotations (plain, comma-separated or list-formatted);
#      defined only for hateful videos; multi-target videos take the group of the first-listed target
import ast
AUDIT.setdefault('hmm_h3_before_fix',
                 pd.crosstab(hatemm_df['label'], hatemm_df['target_group'].fillna(-1)).to_dict())
_GROUP = {'blacks': 0, 'black': 0, 'asian': 0, 'asians': 0, 'whites': 0, 'white': 0,
          'jews': 1, 'jew': 1, 'muslims': 1, 'muslim': 1, 'christians': 1,
          'lgbtq': 2, 'lgbt': 2, 'women': 2, 'sexits': 2, 'sexists': 2,
          'others': 3, 'other': 3}
def _parse_targets(x):
    if pd.isna(x): return []
    s_ = str(x).strip()
    if s_.startswith('['):
        try: items = ast.literal_eval(s_)
        except Exception: items = s_.strip('[]').replace("'", '').split(',')
    else:
        items = s_.split(',')
    return [str(i).strip().lower() for i in items if str(i).strip()]
def _hmm_group(x):
    g = [_GROUP[t] for t in _parse_targets(x) if t in _GROUP]
    return g[0] if g else -1
_tg = hatemm_ann.drop_duplicates('video_file_name').set_index('video_file_name')['target'].map(_hmm_group)
for _d in (hatemm_df, hatemm_train, hatemm_val, hatemm_test):
    _d['target_group'] = _d['video_name'].map(_tg).fillna(-1).astype(int)
    _d.loc[_d['label'] == 0, 'target_group'] = -1
AUDIT['hmm_h3_rule'] = ("parsed plain/comma/list targets; Blacks/Asians/Whites->Race, Jews/Muslims/Christians->Religion, "
                        "LGBTQ/Women/'Sexits'->Gender, Others->Other; first-listed target for multi-target videos; hateful videos only")
print("HateMM label × target_group after fix:\n", pd.crosstab(hatemm_df['label'], hatemm_df['target_group']))

# (C2) UCB harm type: Offensive if the MEAN annotator insult rating >= scale midpoint
_ins = ucb_raw.groupby('comment_id')['insult'].mean()
_lo, _hi = float(ucb_raw['insult'].min()), float(ucb_raw['insult'].max()); _mid = (_lo + _hi) / 2
AUDIT['ucb_insult_scale'] = {'min': _lo, 'max': _hi, 'midpoint': _mid,
                             'per_annotator_counts': ucb_raw['insult'].value_counts().sort_index().to_dict()}
AUDIT.setdefault('ucb_h2_before_fix', ucb_df['harm_type'].value_counts().to_dict())
def _h2(d):
    m = d['comment_id'].map(_ins)
    return np.where(d['label'] == 1, 2, np.where(m >= _mid, 1, 0)).astype(int)
for _d in (ucb_df, ucb_train, ucb_val, ucb_test):
    _d['harm_type'] = _h2(_d)
print(f"UCB insult scale {_lo:g}–{_hi:g}, midpoint {_mid:g};  H2 before: {AUDIT['ucb_h2_before_fix']}  after: {ucb_df['harm_type'].value_counts().to_dict()}")

_h3_names = {0: 'Race/Ethnicity/Origin', 1: 'Religion', 2: 'Gender/Sexuality', 3: 'Other', -1: 'excluded (no label)'}
_h2_names = {0: 'Normal', 1: 'Offensive', 2: 'Hate', -1: 'excluded'}

AUDIT['label_dist'] = {
    'ucb_binary':  ucb_df['label'].value_counts().to_dict(),
    'ucb_h2':      {_h2_names[k]: int(v) for k, v in ucb_df['harm_type'].value_counts().items()},
    'ucb_h3':      {_h3_names[k]: int(v) for k, v in ucb_df['target_group'].value_counts().items()},
    'm3h_binary':  pd.concat([M3H_TR, M3H_VA, M3H_TE])['label'].value_counts().to_dict(),
    'm3h_h3':      {_h3_names[k]: int(v) for k, v in pd.concat([M3H_TR, M3H_VA, M3H_TE])['target_group'].value_counts().items()},
    'hmm_binary':  hatemm_df['label'].value_counts().to_dict(),
    'hmm_h3':      {_h3_names.get(int(k), k): int(v) for k, v in hatemm_df['target_group'].fillna(-1).value_counts().items()},
    'crema_binary': crema_df['label'].value_counts().to_dict(),
}
if 'Category' in train_multi3hate.columns:
    AUDIT['m3h_raw_category_values'] = pd.concat([train_multi3hate, val_multi3hate, test_multi3hate])['Category'].value_counts(dropna=False).to_dict()
if 'target' in hatemm_ann.columns:
    _raw = hatemm_ann['target'].value_counts(dropna=False)
    AUDIT['hmm_raw_target_values'] = _raw.to_dict()
    AUDIT['hmm_raw_targets_unparsed'] = {str(k): int(v) for k, v in _raw.items() if pd.notna(k) and _hmm_group(k) == -1}
    # the annotation file has its own 'label' column -> drop it before merging (avoids label_x / label_y)
    _hate_ann = hatemm_ann.drop(columns=['label'], errors='ignore').merge(
        hatemm_csv[['video_name', 'label']], left_on='video_file_name', right_on='video_name', how='inner')
    AUDIT['hmm_raw_targets_of_hate_videos'] = _hate_ann[_hate_ann['label'] == 1]['target'].value_counts(dropna=False).to_dict()
# UCB binary threshold, as implemented (documented, not changed)
AUDIT['ucb_binary_rule'] = "mean of annotator 'hatespeech' item (0-2) per comment, min-max normalised, label=1 if >=0.5 (i.e. original mean >= 1)"
AUDIT['ucb_h2_rule'] = f"Hate if binary=1; else Offensive if mean annotator 'insult' >= {_mid:g} (midpoint of {_lo:g}-{_hi:g} scale); else Normal"
for k, v in AUDIT['label_dist'].items(): print(f"{k:14s} {v}")
save_audit()
print(f"⏱ cell finished in {(_time.time()-_tc)/60:.1f} min")


In [ ]:
import time as _time; _tc = _time.time()
# ── V3-D: Corrected unified splits + dataset ─────────────────────────────────
SRC2ID = {'m3h': 0, 'ucb': 1, 'hmm': 2, 'crema': 3}

def _txt(v):
    return '' if v is None or (isinstance(v, float) and np.isnan(v)) else str(v)

def make_split(m3h=None, crema=None, crema_embs=None, ucb=None, hmm=None):
    """Builds rows AND the audio-embedding array together (vectorised), so every audio_idx
    points at the correct embedding by construction (fixes the V1/V2 HateMM bug)."""
    parts, embs, n = [], [], 0
    def _col(d, name, default):
        return d[name] if name in d.columns else pd.Series([default] * len(d), index=d.index)
    if m3h is not None:
        cap = _col(m3h, 'Caption', None) if 'Caption' in m3h.columns else _col(m3h, 'text', '')
        parts.append(pd.DataFrame({'text': cap.where(cap.notna(), '').astype(str).values,
                                   'image_path': m3h['image_path'].values, 'audio_idx': -1,
                                   'label': m3h['label'].astype(int).values, 'harm_type': -1,
                                   'target_group': _col(m3h, 'target_group', -1).fillna(-1).astype(int).values,
                                   'source': 'm3h', 'lang': _col(m3h, 'Language', '').fillna('').astype(str).values}))
    if crema is not None:
        assert len(crema) == len(crema_embs)
        parts.append(pd.DataFrame({'text': '', 'image_path': None, 'audio_idx': n + np.arange(len(crema)),
                                   'label': crema['label'].astype(int).values, 'harm_type': -1, 'target_group': -1,
                                   'source': 'crema', 'lang': 'en'}))
        embs.append(np.asarray(crema_embs)); n += len(crema)
    if ucb is not None:
        parts.append(pd.DataFrame({'text': _col(ucb, 'text', '').where(_col(ucb, 'text', '').notna(), '').astype(str).values,
                                   'image_path': None, 'audio_idx': -1, 'label': ucb['label'].astype(int).values,
                                   'harm_type': _col(ucb, 'harm_type', -1).fillna(-1).astype(int).values,
                                   'target_group': _col(ucb, 'target_group', -1).fillna(-1).astype(int).values,
                                   'source': 'ucb', 'lang': 'en'}))
    if hmm is not None:
        vid = hmm['video_id'].astype(str)
        ei = hmm['video_id'].map(hatemm_vid_map).fillna(-1).astype(int).values
        ok = ei >= 0
        ai = np.full(len(hmm), -1); ai[ok] = n + np.arange(ok.sum())
        embs.append(hatemm_embs[ei[ok]]); n += int(ok.sum())
        parts.append(pd.DataFrame({'text': '', 'image_path': (HATEMM_FRAMES_PATH + '/' + vid + '.jpg').values,
                                   'audio_idx': ai, 'label': hmm['label'].astype(int).values, 'harm_type': -1,
                                   'target_group': _col(hmm, 'target_group', -1).fillna(-1).astype(int).values,
                                   'source': 'hmm', 'lang': 'en'}))
    df = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    E = np.vstack(embs).astype(np.float32) if embs else np.zeros((0, AUDIO_EMB_DIM), np.float32)
    return df, E

TRAIN_V3, TRAIN_EMB = make_split(M3H_TR, crema_train, crema_train_embs, ucb_train, hatemm_train)
VAL_V3,   VAL_EMB   = make_split(M3H_VA, crema_val,   crema_val_embs,   ucb_val,   hatemm_val)
TEST_M3H, _         = make_split(m3h=M3H_TE)
TEST_HMM, TEST_HMM_EMB = make_split(hmm=hatemm_test)
TEST_UCB, _         = make_split(ucb=ucb_test)
TEST_SETS = {'m3h': (TEST_M3H, None), 'hmm': (TEST_HMM, TEST_HMM_EMB), 'ucb': (TEST_UCB, None)}

# Image availability audit
def _img_ok(df): 
    m = df.image_path.map(lambda p: isinstance(p, str))
    return f"{int(df.image_path[m].map(os.path.exists).sum())}/{int(m.sum())}"
AUDIT['images_found'] = {'train_m3h': _img_ok(TRAIN_V3[TRAIN_V3.source=='m3h']), 'train_hmm': _img_ok(TRAIN_V3[TRAIN_V3.source=='hmm']),
                         'test_m3h': _img_ok(TEST_M3H), 'test_hmm': _img_ok(TEST_HMM)}
AUDIT['v3_sizes'] = {'train': TRAIN_V3.source.value_counts().to_dict(), 'val': VAL_V3.source.value_counts().to_dict(),
                     'test': {k: len(v[0]) for k, v in TEST_SETS.items()}}
print("Images found:", AUDIT['images_found']); print("Sizes:", AUDIT['v3_sizes'])
save_audit()

class V3Dataset(Dataset):
    """allowed = (text, image, audio) as 0/1. A modality flag is 1 only if the
    modality is allowed AND actually present for that sample."""
    def __init__(self, df, embs, allowed=(1, 1, 1), max_len=MAX_TEXT_LEN):
        self.df = df.reset_index(drop=True); self.embs = embs; self.allowed = allowed; self.ml = max_len
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        txt = r['text'] if isinstance(r['text'], str) else ''
        has_t = int(bool(self.allowed[0]) and len(txt.strip()) > 0)
        enc = xlmr_tok(txt if has_t else '', truncation=True, padding='max_length',
                       max_length=self.ml, return_tensors='pt')
        has_i, pv = 0, torch.zeros(3, 224, 224)
        ip = r['image_path']
        if self.allowed[1] and isinstance(ip, str) and os.path.exists(ip):
            try:
                pv = clip_proc(images=Image.open(ip).convert('RGB'), return_tensors='pt')['pixel_values'].squeeze(0); has_i = 1
            except Exception:
                pass
        has_a, ae = 0, torch.zeros(AUDIO_EMB_DIM)
        ai = int(r['audio_idx'])
        if self.allowed[2] and self.embs is not None and 0 <= ai < len(self.embs):
            ae = torch.from_numpy(np.asarray(self.embs[ai], dtype=np.float32)); has_a = 1
        return {'input_ids': enc['input_ids'].squeeze(0), 'attention_mask': enc['attention_mask'].squeeze(0),
                'pixel_values': pv, 'audio_emb': ae,
                'flags': torch.tensor([has_t, has_i, has_a], dtype=torch.float),
                'label': torch.tensor(int(r['label'])), 'harm_type': torch.tensor(int(r['harm_type'])),
                'target_group': torch.tensor(int(r['target_group']))}
print("V3 splits + dataset ready ✓")
print(f"⏱ cell finished in {(_time.time()-_tc)/60:.1f} min")


In [ ]:
import time as _time; _tc = _time.time()
# ── V3-E: Model — seven fusion variants under one class ──────────────────────
#  tcaf_original : faithful V1 behaviour (text always encoded, no mask, 3×512 concat)
#  tcaf_masked   : TCAF with key-padding mask; mean over PRESENT tokens  (main model)
#  mean_masked   : identical tokens + pooling, NO attention  -> isolates cross-modal attention
#  gating_masked : learned softmax gate over present tokens (prior-work style)
#  late_masked   : one binary classifier per modality, logits averaged over present modalities
#  mlca_masked   : re-implemented MLCA [50] (3 stacked cross-modal MHA levels), masked
#  cmfusion_masked: re-implemented CMFusion [51] (channel SE + modality gate), masked
FUSIONS = ('tcaf_original', 'tcaf_masked', 'mean_masked', 'gating_masked', 'late_masked', 'mlca_masked', 'cmfusion_masked')

def _fresh_encoders_v3():
    x = AutoModel.from_pretrained('xlm-roberta-base')
    for p in x.parameters(): p.requires_grad = False
    for l in x.encoder.layer[-3:]:
        for p in l.parameters(): p.requires_grad = True
    if getattr(x, 'pooler', None) is not None:
        for p in x.pooler.parameters(): p.requires_grad = True
    c = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').vision_model
    for p in c.parameters(): p.requires_grad = False
    for l in c.encoder.layers[-2:]:
        for p in l.parameters(): p.requires_grad = True
    return x, c

class MLCAMasked(nn.Module):
    def __init__(self, d=512, nhead=8, levels=3, dropout=0.1):
        super().__init__()
        self.attn = nn.ModuleList([nn.MultiheadAttention(d, nhead, dropout=dropout, batch_first=True) for _ in range(levels)])
        self.norm = nn.ModuleList([nn.LayerNorm(d) for _ in range(levels)])
    def forward(self, x, kpm):
        for a, n in zip(self.attn, self.norm):
            h, _ = a(x, x, x, key_padding_mask=kpm, need_weights=False)
            x = n(x + h)
        return x

class CMFusionMasked(nn.Module):
    def __init__(self, d=512, reduction=8):
        super().__init__()
        r = max(d // reduction, 8)
        self.se = nn.Sequential(nn.Linear(d, r), nn.ReLU(), nn.Linear(r, d), nn.Sigmoid())
        self.gate = nn.Linear(3 * d, 3)
    def forward(self, x, present):
        x = x * self.se(x)
        s = self.gate(x.reshape(x.size(0), -1)).float().masked_fill(~present, -1e4)
        w = torch.softmax(s, 1).unsqueeze(-1) * present.float().unsqueeze(-1)
        return (x * w).sum(1)

def _head(fd, c):
    return nn.Sequential(nn.Linear(fd, 256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256, c))

class TCAFBlockV3(nn.Module):
    def __init__(self, d=512, nhead=8, layers=2, dropout=0.1):
        super().__init__()
        layer = nn.TransformerEncoderLayer(d, nhead, d * 4, dropout, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
    def forward(self, x, kpm=None):
        return self.norm(self.enc(x, src_key_padding_mask=kpm))

class V3Net(nn.Module):
    def __init__(self, fusion, d=HIDDEN_DIM, adim=None, enc_factory=None):
        super().__init__()
        assert fusion in FUSIONS, fusion
        self.fusion_type, self.d = fusion, d
        adim = adim or AUDIO_EMB_DIM
        self.text_encoder, self.image_encoder = (enc_factory or _fresh_encoders_v3)()
        mk = lambda i: nn.Sequential(nn.Linear(i, d), nn.GELU(), nn.Dropout(0.1), nn.LayerNorm(d))
        self.text_proj  = mk(self.text_encoder.config.hidden_size)
        self.image_proj = mk(self.image_encoder.config.hidden_size)
        self.audio_proj = mk(adim)
        self.mod_embed  = nn.Parameter(torch.randn(1, 3, d) * 0.02)
        if fusion in ('tcaf_original', 'tcaf_masked'): self.fusion = TCAFBlockV3(d)
        if fusion == 'gating_masked': self.gate = nn.Linear(d, 1)
        if fusion == 'late_masked':   self.uni_heads = nn.ModuleList([_head(d, 2) for _ in range(3)])
        if fusion == 'mlca_masked':   self.fusion = MLCAMasked(d)
        if fusion == 'cmfusion_masked': self.fusion = CMFusionMasked(d)
        fd = 3 * d if fusion == 'tcaf_original' else d
        self.drop = nn.Dropout(0.2)
        self.head_binary, self.head_type, self.head_target = _head(fd, 2), _head(fd, 3), _head(fd, 4)

    def _fill(self, on, fn, B, dev):
        if on.any():
            v = fn(on).float()
            out = v.new_zeros(B, v.size(-1)); out[on] = v
            return out
        return torch.zeros(B, self.d, device=dev)

    def forward(self, ids, am, pv, ae, flags):
        B, dev = ids.size(0), ids.device
        t_on = torch.ones(B, dtype=torch.bool, device=dev) if self.fusion_type == 'tcaf_original' else flags[:, 0] > 0
        i_on, a_on = flags[:, 1] > 0, flags[:, 2] > 0
        # Encoders run only on samples where the modality is present (numerically identical, much faster)
        tf  = self._fill(t_on, lambda m: self.text_proj(self.text_encoder(input_ids=ids[m], attention_mask=am[m]).last_hidden_state[:, 0]), B, dev)
        imf = self._fill(i_on, lambda m: self.image_proj(self.image_encoder(pixel_values=pv[m]).last_hidden_state[:, 0]), B, dev)
        af  = self._fill(a_on, lambda m: self.audio_proj(ae[m]), B, dev)
        x = torch.stack([tf, imf, af], 1) + self.mod_embed
        present = torch.stack([t_on, i_on, a_on], 1)
        pf = present.float().unsqueeze(-1); cnt = pf.sum(1).clamp(min=1.0)
        ft, lb_late = self.fusion_type, None
        if ft == 'tcaf_original':
            pooled = self.fusion(x).reshape(B, -1)
        elif ft == 'tcaf_masked':
            kpm = ~present
            kpm = kpm & ~kpm.all(1, keepdim=True)          # no-modality rows: avoid all-masked NaN
            pooled = (self.fusion(x, kpm) * pf).sum(1) / cnt
        elif ft == 'mean_masked':
            pooled = (x * pf).sum(1) / cnt
        elif ft == 'gating_masked':
            s = self.gate(x).squeeze(-1).float().masked_fill(~present, -1e4)
            w = torch.softmax(s, 1).unsqueeze(-1) * pf
            pooled = (x * w).sum(1)
        elif ft == 'mlca_masked':
            kpm = ~present; kpm = kpm & ~kpm.all(1, keepdim=True)
            pooled = (self.fusion(x, kpm) * pf).sum(1) / cnt
        elif ft == 'cmfusion_masked':
            pooled = self.fusion(x, present)
        else:  # late_masked
            pooled = (x * pf).sum(1) / cnt
            per = torch.stack([self.uni_heads[m](x[:, m]) for m in range(3)], 1).float()
            lb_late = (per * pf).sum(1) / cnt
        pooled = self.drop(pooled)
        lb = self.head_binary(pooled) if lb_late is None else lb_late
        return lb, self.head_type(pooled), self.head_target(pooled)

def loss_v3(lb, lt, ltg, y, ht, tg, cw):
    L1 = F.cross_entropy(lb.float(), y, weight=cw)
    m2, m3 = ht >= 0, tg >= 0
    L2 = F.cross_entropy(lt.float()[m2], ht[m2]) if m2.any() else lb.float().sum() * 0
    L3 = F.cross_entropy(ltg.float()[m3], tg[m3]) if m3.any() else lb.float().sum() * 0
    return W_B3 * L1 + W_T3 * L2 + W_G3 * L3
print("V3Net defined ✓  fusions:", FUSIONS)
print(f"⏱ cell finished in {(_time.time()-_tc)/60:.1f} min")


In [ ]:
import time as _time; _tc = _time.time()
# ── V3-F: Metrics, bootstrap, prediction + training runner ───────────────────
def m_bin(t, p):
    t, p = np.asarray(t), np.asarray(p); yh = (p > 0.5).astype(int)
    if len(np.unique(t)) < 2: return None
    return {'N': int(len(t)), 'Acc': float(accuracy_score(t, yh)), 'F1': float(f1_score(t, yh, zero_division=0)),
            'F1_macro': float(f1_score(t, yh, average='macro', zero_division=0)),
            'AUC': float(roc_auc_score(t, p)), 'AP': float(average_precision_score(t, p)),
            'MCC': float(matthews_corrcoef(t, yh))}

def boot_ci(t, p, B=N_BOOT, seed=0):
    t, p = np.asarray(t), np.asarray(p); rng = np.random.default_rng(seed); n = len(t)
    acc = {'AUC': [], 'MCC': [], 'F1': []}
    for _ in range(B):
        ix = rng.integers(0, n, n); tt, pp = t[ix], p[ix]
        if len(np.unique(tt)) < 2: continue
        yh = (pp > 0.5).astype(int)
        acc['AUC'].append(roc_auc_score(tt, pp)); acc['MCC'].append(matthews_corrcoef(tt, yh))
        acc['F1'].append(f1_score(tt, yh, zero_division=0))
    return {k: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] for k, v in acc.items()}

def _to_dev(b):
    return [b[k].to(device, non_blocking=True) for k in ('input_ids', 'attention_mask', 'pixel_values', 'audio_emb', 'flags')]

def _predict(model, df, embs, allowed, bs=64):
    ldr = DataLoader(V3Dataset(df, embs, allowed), batch_size=bs, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    model.eval(); out = {k: [] for k in ('true', 'prob', 'h2_true', 'h2_pred', 'h3_true', 'h3_pred', 'n_present')}
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16, enabled=USE_AMP):
        for b in ldr:
            lb, lt, ltg = model(*_to_dev(b))
            out['prob'].append(torch.softmax(lb.float(), 1)[:, 1].cpu().numpy())
            out['true'].append(b['label'].numpy())
            out['h2_true'].append(b['harm_type'].numpy()); out['h2_pred'].append(lt.argmax(1).cpu().numpy())
            out['h3_true'].append(b['target_group'].numpy()); out['h3_pred'].append(ltg.argmax(1).cpu().numpy())
            out['n_present'].append(b['flags'].sum(1).numpy())
    return {k: np.concatenate(v) for k, v in out.items()}

def _aux(pr):
    res = {}
    for h in ('h2', 'h3'):
        m = pr[f'{h}_true'] >= 0
        if m.sum() > 0:
            t, p = pr[f'{h}_true'][m], pr[f'{h}_pred'][m]
            res[h.upper()] = {'N': int(m.sum()), 'Acc': float(accuracy_score(t, p)),
                              'F1_macro': float(f1_score(t, p, average='macro', zero_division=0))}
    return res

def run_v3(name, fusion, seed, allowed=('text', 'image', 'audio'), exclude=(), enc_factory=None, freeze_encoders=False):
    tag = f'{name}__s{seed}'; rp = f'{V3_DIR}/runs/{tag}.json'
    if os.path.exists(rp) and not FORCE_RETRAIN_V3:
        print(f"[cached] {tag}"); return json.load(open(rp))
    print(f"\n{'='*72}\n  RUN {tag}   fusion={fusion}  modalities={allowed}  exclude={exclude}\n{'='*72}")
    t0 = time.time(); set_seed(seed)
    al = tuple(int(m in allowed) for m in ('text', 'image', 'audio'))
    tr = TRAIN_V3[~TRAIN_V3.source.isin(exclude)].reset_index(drop=True)
    va = VAL_V3[~VAL_V3.source.isin(exclude)].reset_index(drop=True)
    cw = torch.tensor(compute_class_weight('balanced', classes=np.array([0, 1]), y=tr.label.values),
                      dtype=torch.float, device=device)
    g = torch.Generator(); g.manual_seed(seed)
    tl = DataLoader(V3Dataset(tr, TRAIN_EMB, al), batch_size=BATCH_V3, shuffle=True, num_workers=NUM_WORKERS,
                    pin_memory=True, generator=g, worker_init_fn=lambda w: np.random.seed(seed + w),
                    persistent_workers=NUM_WORKERS > 0)
    model = V3Net(fusion, enc_factory=enc_factory).to(device)
    if freeze_encoders:
        for p in list(model.text_encoder.parameters()) + list(model.image_encoder.parameters()): p.requires_grad = False
    params = [p for p in model.parameters() if p.requires_grad]
    keep = {n for n, p in model.named_parameters() if p.requires_grad} | \
           {k for k in model.state_dict() if not k.startswith(('text_encoder.', 'image_encoder.'))}
    opt = optim.AdamW(params, lr=LR_V3, weight_decay=0.01)
    sch = optim.lr_scheduler.OneCycleLR(opt, max_lr=LR_V3, steps_per_epoch=max(1, len(tl)), epochs=EPOCHS_V3, pct_start=0.1)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    best, best_state, bad, hist = -1.0, None, 0, []
    for ep in range(EPOCHS_V3):
        model.train(); tot, nb = 0.0, 0
        for b in tqdm(tl, desc=f'{tag} ep{ep+1}/{EPOCHS_V3}', leave=False):
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16, enabled=USE_AMP):
                lb, lt, ltg = model(*_to_dev(b))
            loss = loss_v3(lb, lt, ltg, b['label'].to(device), b['harm_type'].to(device), b['target_group'].to(device), cw)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(params, 1.0); scaler.step(opt); scaler.update(); sch.step()
            tot += loss.item(); nb += 1
        vr = _predict(model, va, VAL_EMB, al)
        vf1 = float(f1_score(vr['true'], (vr['prob'] > 0.5).astype(int), zero_division=0))
        hist.append({'epoch': ep + 1, 'train_loss': tot / max(nb, 1), 'val_f1': vf1})
        print(f"  epoch {ep+1}: loss={tot/max(nb,1):.4f}  val F1={vf1:.4f}")
        if vf1 > best:
            best, bad = vf1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in keep}
        else:
            bad += 1
            if bad >= PATIENCE_V3: print("  early stop"); break
    model.load_state_dict(best_state, strict=False)
    if name in SAVE_CKPT_FOR: torch.save(best_state, f'{V3_DIR}/ckpt/{tag}.pt')

    res = {'name': name, 'fusion': fusion, 'seed': seed, 'allowed': list(allowed), 'exclude': list(exclude),
           'frozen_encoders': freeze_encoders, 'n_trainable': int(sum(p.numel() for p in params)),
           'best_val_f1': best, 'history': hist, 'test': {}, 'aux': {}, 'per_language': {}}
    for bench, (df, embs) in TEST_SETS.items():
        pr = _predict(model, df, embs, al)
        np.savez(f'{V3_DIR}/preds/{tag}__{bench}.npz', **pr)
        if (pr['n_present'] > 0).mean() == 0:           # benchmark has none of the allowed modalities
            res['test'][bench] = None; continue
        m = m_bin(pr['true'], pr['prob'])
        if m is not None: m['CI95'] = boot_ci(pr['true'], pr['prob'], seed=seed)
        res['test'][bench] = m
        res['aux'][bench] = _aux(pr)
        if bench == 'm3h':
            for lg in sorted(df['lang'].unique()):
                ix = (df['lang'] == lg).values
                res['per_language'][lg] = m_bin(pr['true'][ix], pr['prob'][ix])
    res['minutes'] = round((time.time() - t0) / 60, 1)
    json.dump(res, open(rp, 'w'), indent=2)
    for bench, m in res['test'].items():
        if m: print(f"  TEST {bench}: F1={m['F1']:.4f}  AUC={m['AUC']:.4f}  MCC={m['MCC']:.4f}")
    print(f"  done in {res['minutes']} min")
    del model, opt, best_state; torch.cuda.empty_cache()
    return res
print("run_v3() ready ✓")
print(f"⏱ cell finished in {(_time.time()-_tc)/60:.1f} min")


In [ ]:
import time as _time; _tc = _time.time()
# ── V3-G: Experiment plan (tiered; seeds run outer so every config gets seed 42 first) ──
ALL = ('text', 'image', 'audio')
# (name, fusion, allowed modalities, excluded training sources, freeze encoders)
EXPERIMENTS = {
    # Tier 1 — R3.2 (masking vs. original) and R3.4 (attention isolated against identical no-attention model)
    1: [('tcaf_masked',   'tcaf_masked',   ALL, (), False),
        ('mean_masked',   'mean_masked',   ALL, (), False),
        ('gating_masked', 'gating_masked', ALL, (), False),
        ('tcaf_original', 'tcaf_original', ALL, (), False)],
    # Tier 2 — R3.1 (same-protocol baselines on the same test splits), R3.4 (does audio help?), R3.5 (CREMA-D)
    2: [('late_masked',          'late_masked', ALL,               (), False),
        ('text_only',            'mean_masked', ('text',),         (), False),
        ('image_only',           'mean_masked', ('image',),        (), False),
        ('audio_only',           'mean_masked', ('audio',),        (), False),
        ('tcaf_masked_no_audio', 'tcaf_masked', ('text', 'image'), (), False),
        ('tcaf_masked_no_crema', 'tcaf_masked', ALL,               ('crema',), False)],
    # Tier 3 — round-1 commitments re-run on the corrected pipeline (MLCA [50], CMFusion [51])
    3: [('mlca_masked',     'mlca_masked',     ALL, (), False),
        ('cmfusion_masked', 'cmfusion_masked', ALL, (), False)],
    # Tier 4 — supporting (single seed): retrained w/o text; fully frozen encoders (contribution 6)
    4: [('tcaf_masked_no_text', 'tcaf_masked', ('image', 'audio'), (), False),
        ('tcaf_masked_frozen',  'tcaf_masked', ALL,                (), True)],
}
TIER_SEEDS = {1: SEEDS, 2: SEEDS, 3: SEEDS, 4: SEEDS[:1]}
RUN_TIERS = [1, 2, 3, 4]   # ~38 runs × ~12 min ≈ 7.5 h; cached runs are skipped if the session restarts

for tier in RUN_TIERS:
    for seed in TIER_SEEDS[tier]:
        for name, fus, al, ex, frz in EXPERIMENTS[tier]:
            try:
                run_v3(name, fus, seed, al, ex, freeze_encoders=frz)
            except Exception:
                print(f"!! {name} seed {seed} failed:"); traceback.print_exc()
print("\nAll requested V3 runs finished (or cached) ✓")
print(f"⏱ cell finished in {(_time.time()-_tc)/60:.1f} min")


In [ ]:
import time as _time; _tc = _time.time()
# ── V3-H: Aggregate — mean ± std over seeds, paired bootstrap between configs ─
runs = [json.load(open(f)) for f in sorted(glob.glob(f'{V3_DIR}/runs/*.json'))]
_rows = []
for r in runs:
    for bench, m in r['test'].items():
        if m: _rows.append({'config': r['name'], 'seed': r['seed'], 'bench': bench,
                            **{k: m[k] for k in ('Acc', 'F1', 'F1_macro', 'AUC', 'AP', 'MCC')},
                            'AUC_lo': m['CI95']['AUC'][0], 'AUC_hi': m['CI95']['AUC'][1],
                            'MCC_lo': m['CI95']['MCC'][0], 'MCC_hi': m['CI95']['MCC'][1]})
per_seed = pd.DataFrame(_rows)
per_seed.to_csv(f'{V3_DIR}/tables/v3_per_seed.csv', index=False)
summary = per_seed.groupby(['config', 'bench'])[['Acc', 'F1', 'F1_macro', 'AUC', 'AP', 'MCC']].agg(['mean', 'std', 'count']).round(4)
summary.to_csv(f'{V3_DIR}/tables/v3_summary_mean_std.csv')
print("=" * 90); print("  V3 SUMMARY — mean ± std over seeds (test sets)"); print("=" * 90)
display(summary)

def _load(cfg, seed, bench):
    p = f'{V3_DIR}/preds/{cfg}__s{seed}__{bench}.npz'
    return np.load(p) if os.path.exists(p) else None

def paired_boot(t, pa, pb, B=N_BOOT, seed=0):
    rng = np.random.default_rng(seed); n = len(t); dA, dM = [], []
    for _ in range(B):
        ix = rng.integers(0, n, n); tt = t[ix]
        if len(np.unique(tt)) < 2: continue
        dA.append(roc_auc_score(tt, pa[ix]) - roc_auc_score(tt, pb[ix]))
        dM.append(matthews_corrcoef(tt, (pa[ix] > .5).astype(int)) - matthews_corrcoef(tt, (pb[ix] > .5).astype(int)))
    q = lambda v: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))]
    return q(dA), q(dM), float(np.mean(np.array(dA) > 0)), float(np.mean(np.array(dM) > 0))

PAIRS = [('tcaf_masked', b) for b in ('mean_masked', 'gating_masked', 'late_masked', 'mlca_masked', 'cmfusion_masked',
                                      'tcaf_original', 'tcaf_masked_no_audio', 'tcaf_masked_no_crema', 'image_only',
                                      'text_only', 'audio_only', 'tcaf_masked_no_text', 'tcaf_masked_frozen')]
_ALL_SEEDS = sorted({r['seed'] for r in runs})
_prow = []
for a, b in PAIRS:
    for bench in ('m3h', 'hmm', 'ucb'):
        ens_a, ens_b, t_ref = [], [], None
        for s in _ALL_SEEDS:
            A, Bp = _load(a, s, bench), _load(b, s, bench)
            if A is None or Bp is None: continue
            if (Bp['n_present'] > 0).mean() == 0 or (A['n_present'] > 0).mean() == 0: continue
            t_ref = A['true']; ens_a.append(A['prob']); ens_b.append(Bp['prob'])
            ciA, ciM, pA, pM = paired_boot(A['true'], A['prob'], Bp['prob'], seed=s)
            _prow.append({'A': a, 'B': b, 'bench': bench, 'seed': s,
                          'dAUC': roc_auc_score(A['true'], A['prob']) - roc_auc_score(A['true'], Bp['prob']),
                          'dAUC_CI': ciA, 'P(dAUC>0)': pA,
                          'dMCC': matthews_corrcoef(A['true'], (A['prob'] > .5).astype(int)) - matthews_corrcoef(A['true'], (Bp['prob'] > .5).astype(int)),
                          'dMCC_CI': ciM, 'P(dMCC>0)': pM})
        if len(ens_a) >= 2:   # seed-ensemble comparison (averaged probabilities)
            pa, pb = np.mean(ens_a, 0), np.mean(ens_b, 0)
            ciA, ciM, pA, pM = paired_boot(t_ref, pa, pb, seed=99)
            _prow.append({'A': a, 'B': b, 'bench': bench, 'seed': 'ensemble',
                          'dAUC': roc_auc_score(t_ref, pa) - roc_auc_score(t_ref, pb), 'dAUC_CI': ciA, 'P(dAUC>0)': pA,
                          'dMCC': matthews_corrcoef(t_ref, (pa > .5).astype(int)) - matthews_corrcoef(t_ref, (pb > .5).astype(int)),
                          'dMCC_CI': ciM, 'P(dMCC>0)': pM})
paired = pd.DataFrame(_prow)
paired.to_csv(f'{V3_DIR}/tables/v3_paired_comparisons.csv', index=False)
print("\n" + "=" * 90); print("  PAIRED COMPARISONS (Δ = A − B; 95% paired-bootstrap CI)"); print("=" * 90)
display(paired.round(4))

# Per-language (Multi3Hate) and auxiliary heads
_pl = [{'config': r['name'], 'seed': r['seed'], 'lang': lg, **m} for r in runs for lg, m in r['per_language'].items() if m]
pd.DataFrame(_pl).to_csv(f'{V3_DIR}/tables/v3_per_language.csv', index=False)
_ax = [{'config': r['name'], 'seed': r['seed'], 'bench': b, 'head': h, **v}
       for r in runs for b, hs in r['aux'].items() for h, v in hs.items()]
pd.DataFrame(_ax).to_csv(f'{V3_DIR}/tables/v3_aux_heads.csv', index=False)
if _pl:
    display(pd.DataFrame(_pl).groupby(['config', 'lang'])[['N', 'F1', 'AUC', 'MCC']].mean().round(4))
print(f"⏱ cell finished in {(_time.time()-_tc)/60:.1f} min")


### V3-J: Explainability, rationale plausibility, adversarial robustness (inference only)

Run this after Tier 1 has produced `ckpt/tcaf_masked__s42.pt`. It replaces the old-pipeline Table 9, Table 10 and Fig. 15 with:
1. **J1**: SHAP + LIME on the same example sentences used in V9 (Table 10 / Fig. 15a).
2. **J2**: **Rationale plausibility on HateXplain**, which answers R3.3 with measurement instead of a claim. On held-out HateXplain test posts (a corpus never used in training), we score how well each attribution method's word ranking matches human rationales, using ERASER-style token-level AUPRC and top-k token F1, against a random baseline.
3. **J3**: DeepWordBug + TextFooler on 100 held-out UCB **test** hate comments, with standard definitions (ASR = successes / (successes + failures); misclassified samples are reported as skipped, never counted as robust).

In [ ]:
import time as _time; _tc = _time.time()
# ── V3-J0: Load the main checkpoint (eager attention so attentions are returned) ──
XAI_SEED = 42
XAI_DIR = f'{V3_DIR}/xai'; os.makedirs(XAI_DIR, exist_ok=True)
N_PLAUS, LIME_SAMPLES, N_ATTACK = 300, 300, 100

def _eager_encoders():
    x = AutoModel.from_pretrained('xlm-roberta-base', attn_implementation='eager')
    c = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').vision_model
    return x, c

xai_model = V3Net('tcaf_masked', enc_factory=_eager_encoders).to(device)
_sd = torch.load(f'{V3_DIR}/ckpt/tcaf_masked__s{XAI_SEED}.pt', map_location=device)
_missing, _unexpected = xai_model.load_state_dict(_sd, strict=False)
assert not _unexpected, _unexpected
assert all(k.startswith(('text_encoder.', 'image_encoder.')) for k in _missing)   # only frozen pretrained weights
xai_model.eval()

def _text_only_logits(ids=None, am=None, embeds=None):
    """Text-only path of tcaf_masked (flags = [1,0,0]); accepts inputs_embeds for gradients."""
    m = xai_model
    h = m.text_encoder(input_ids=ids, inputs_embeds=embeds, attention_mask=am).last_hidden_state[:, 0]
    tf = m.text_proj(h).float(); z = torch.zeros_like(tf)
    x = torch.stack([tf, z, z], 1) + m.mod_embed
    present = torch.tensor([True, False, False], device=tf.device).unsqueeze(0).expand(tf.size(0), 3)
    pooled = (m.fusion(x, ~present) * present.float().unsqueeze(-1)).sum(1)
    return m.head_binary(m.drop(pooled))

@torch.no_grad()
def p_hate_texts(texts, bs=64):
    out = []
    for i in range(0, len(texts), bs):
        enc = xlmr_tok([str(t) for t in texts[i:i + bs]], truncation=True, padding='max_length',
                       max_length=MAX_TEXT_LEN, return_tensors='pt')
        lg = _text_only_logits(enc['input_ids'].to(device), enc['attention_mask'].to(device))
        out.append(torch.softmax(lg.float(), 1).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, 2))

# Consistency check: helper path == full model forward with flags [1,0,0]
_enc = xlmr_tok(['consistency check sentence'], truncation=True, padding='max_length', max_length=MAX_TEXT_LEN, return_tensors='pt')
with torch.no_grad():
    _a = _text_only_logits(_enc['input_ids'].to(device), _enc['attention_mask'].to(device))
    _b = xai_model(_enc['input_ids'].to(device), _enc['attention_mask'].to(device), torch.zeros(1, 3, 224, 224, device=device),
                   torch.zeros(1, AUDIO_EMB_DIM, device=device), torch.tensor([[1., 0., 0.]], device=device))[0]
assert torch.allclose(_a.float(), _b.float(), atol=1e-4), (_a, _b)
print("XAI model loaded; text-only helper matches full forward ✓")
print(f"⏱ cell finished in {(_time.time()-_tc)/60:.1f} min")


In [ ]:
import time as _time; _tc = _time.time()
# ── V3-J1: SHAP + LIME on the V9 example sentences (regenerates Table 10 / Fig. 15a) ──
from lime.lime_text import LimeTextExplainer
XAI_TEXTS = ['Those immigrants are destroying our culture and society.',
             'She plays football brilliantly, amazing match tonight!',
             'People of that religion should not be allowed here.',
             'I disagree with their political views but respect their rights.']
_ph = p_hate_texts(XAI_TEXTS)[:, 1]

_masker = shap.maskers.Text(xlmr_tok)
_expl = shap.Explainer(lambda t: p_hate_texts(list(t)), _masker, output_names=['Not Hate', 'Hate'])
_sv = _expl(XAI_TEXTS, batch_size=8)
_lime = LimeTextExplainer(class_names=['Not Hate', 'Hate'], random_state=42)

qual_rows = []
fig, axes = plt.subplots(len(XAI_TEXTS), 2, figsize=(13, 3.2 * len(XAI_TEXTS)))
for i, txt in enumerate(XAI_TEXTS):
    toks, vals = list(_sv[i].data), np.asarray(_sv[i].values[:, 1])
    keep = [j for j, t in enumerate(toks) if str(t).strip()]
    toks, vals = [str(toks[j]).strip() for j in keep], vals[keep]
    ex = _lime.explain_instance(txt, lambda T: p_hate_texts(list(T)), labels=[1], num_features=10, num_samples=1000)
    lw = ex.as_list(label=1)
    qual_rows.append({'text': txt, 'p_hate': float(_ph[i]), 'predicted': 'Hate' if _ph[i] > 0.5 else 'Not hate',
                      'shap_top_pos': toks[int(np.argmax(vals))] if len(vals) else '', 'shap_top_pos_val': float(np.max(vals)) if len(vals) else 0,
                      'lime_top': max(lw, key=lambda z: abs(z[1]))[0] if lw else '', 'lime_top_val': max(lw, key=lambda z: abs(z[1]))[1] if lw else 0,
                      'shap_tokens': json.dumps(list(zip(toks, [round(float(v), 4) for v in vals]))),
                      'lime_words': json.dumps([(w, round(float(v), 4)) for w, v in lw])})
    for ax, (labels, v, ttl) in zip(axes[i], [(toks, vals, 'SHAP'), ([w for w, _ in lw], np.array([v for _, v in lw]), 'LIME')]):
        ax.barh(range(len(v)), v, color=['#c0392b' if x > 0 else '#2471a3' for x in v])
        ax.set_yticks(range(len(v))); ax.set_yticklabels(labels, fontsize=8); ax.invert_yaxis()
        ax.axvline(0, color='k', lw=0.6); ax.set_title(f'{ttl} — ex. {i+1}  (p(hate)={_ph[i]:.3f})', fontsize=9)
plt.tight_layout(); plt.savefig(f'{XAI_DIR}/fig_shap_lime_examples.png', dpi=300); plt.show()
pd.DataFrame(qual_rows).to_csv(f'{XAI_DIR}/table10_shap_lime_examples.csv', index=False)
display(pd.DataFrame(qual_rows)[['text', 'p_hate', 'predicted', 'shap_top_pos', 'shap_top_pos_val', 'lime_top', 'lime_top_val']])
print(f"⏱ cell finished in {(_time.time()-_tc)/60:.1f} min")


In [ ]:
import time as _time; _tc = _time.time()
# ── V3-J2: Rationale plausibility on HateXplain test posts (ERASER-style) ─────
import urllib.request
_HX  = 'https://raw.githubusercontent.com/hate-alert/HateXplain/master/Data/dataset.json'
_DIV = 'https://raw.githubusercontent.com/hate-alert/HateXplain/master/Data/post_id_divisions.json'
hx  = json.load(urllib.request.urlopen(_HX)); hx_div = json.load(urllib.request.urlopen(_DIV))

def build_hx_rows(hx, ids):
    rows = []
    for pid in ids:
        d = hx[pid]; labs = [a['label'] for a in d['annotators']]
        maj = max(set(labs), key=labs.count)
        if labs.count(maj) < 2 or maj == 'normal' or not d.get('rationales'): continue
        R = np.array(d['rationales'], dtype=float)
        if R.ndim != 2 or R.shape[1] != len(d['post_tokens']): continue
        gold = (R.mean(0) >= 0.5).astype(int)            # majority of rationale-providing annotators
        if gold.sum() == 0 or gold.sum() == len(gold): continue
        rows.append({'post_id': pid, 'tokens': d['post_tokens'], 'gold': gold, 'label': maj})
    return pd.DataFrame(rows)

hx_df = build_hx_rows(hx, hx_div['test'])
AUDIT['hatexplain_eligible_test_posts'] = len(hx_df)
hx_df = hx_df.sample(min(N_PLAUS, len(hx_df)), random_state=42).reset_index(drop=True)
print(f"HateXplain test posts with hateful/offensive majority + non-trivial rationale: {AUDIT['hatexplain_eligible_test_posts']}  → using {len(hx_df)}")

def _enc_words(words):
    return xlmr_tok(words, is_split_into_words=True, truncation=True, max_length=MAX_TEXT_LEN, return_tensors='pt')

def _to_words(sub, enc, n):
    out = np.zeros(n)
    for j, w in enumerate(enc.word_ids(0)):
        if w is not None: out[w] += float(sub[j])
    return out

def attn_scores(words):
    enc = _enc_words(words)
    with torch.no_grad():
        o = xai_model.text_encoder(input_ids=enc['input_ids'].to(device), attention_mask=enc['attention_mask'].to(device), output_attentions=True)
    return _to_words(o.attentions[-1][0].mean(0)[0].float().cpu().numpy(), enc, len(words))

def gxi_scores(words):
    enc = _enc_words(words)
    ids, am = enc['input_ids'].to(device), enc['attention_mask'].to(device)
    e = xai_model.text_encoder.embeddings.word_embeddings(ids).detach().requires_grad_(True)
    _text_only_logits(am=am, embeds=e)[0, 1].backward()
    return _to_words((e.grad * e).sum(-1).abs()[0].detach().cpu().numpy(), enc, len(words))

_lime_pos = LimeTextExplainer(class_names=['Not Hate', 'Hate'], bow=False, split_expression=lambda s: s.split(' '),
                              mask_string='<unk>', random_state=42)
def lime_scores(words):
    ex = _lime_pos.explain_instance(' '.join(words), lambda T: p_hate_texts(list(T)), labels=[1],
                                    num_features=len(words), num_samples=LIME_SAMPLES)
    m = dict(ex.as_map()[1]); return np.array([m.get(i, 0.0) for i in range(len(words))])

def plaus(scores, gold):
    ap = average_precision_score(gold, scores)
    k = int(gold.sum()); pred = np.zeros_like(gold); pred[np.argsort(-scores, kind='stable')[:k]] = 1
    return ap, 2 * int((pred & gold).sum()) / (int(pred.sum()) + k)

_rng = np.random.default_rng(42)
METHODS = {'Attention (CLS, last layer)': attn_scores, 'Gradient × Input': gxi_scores, 'LIME': lime_scores,
           'Random baseline': lambda w: _rng.random(len(w))}
_p = p_hate_texts([' '.join(t) for t in hx_df.tokens])[:, 1]
hx_df['p_hate'] = _p; hx_df['pred_hate'] = (_p > 0.5).astype(int)
plaus_rows = []
t0 = time.time()
for i, r in hx_df.iterrows():
    for mname, fn in METHODS.items():
        ap, f1 = plaus(fn(r.tokens), r.gold)
        plaus_rows.append({'post_id': r.post_id, 'method': mname, 'AUPRC': ap, 'TokenF1_topk': f1, 'pred_hate': r.pred_hate})
    if i % 50 == 0: print(f"  {i}/{len(hx_df)}  ({time.time()-t0:.0f}s)")
plaus_df = pd.DataFrame(plaus_rows); plaus_df.to_csv(f'{XAI_DIR}/plausibility_per_post.csv', index=False)

def _mean_ci(v, B=2000, seed=0):
    v = np.asarray(v); rng = np.random.default_rng(seed)
    bs = [v[rng.integers(0, len(v), len(v))].mean() for _ in range(B)]
    return float(v.mean()), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))
_tab = []
for mname, g in plaus_df.groupby('method', sort=False):
    for subset, gg in [('all', g), ('predicted hate', g[g.pred_hate == 1]), ('predicted not-hate', g[g.pred_hate == 0])]:
        if len(gg) == 0: continue
        a, alo, ahi = _mean_ci(gg.AUPRC); f, flo, fhi = _mean_ci(gg.TokenF1_topk)
        _tab.append({'method': mname, 'subset': subset, 'N': len(gg), 'AUPRC': a, 'AUPRC_CI': f'[{alo:.3f}, {ahi:.3f}]',
                     'TokenF1': f, 'TokenF1_CI': f'[{flo:.3f}, {fhi:.3f}]'})
plaus_tab = pd.DataFrame(_tab); plaus_tab.to_csv(f'{XAI_DIR}/table_plausibility_hatexplain.csv', index=False)
AUDIT['hatexplain_model_predicts_hate_rate'] = float(hx_df.pred_hate.mean())
print(f"Model flags {100*hx_df.pred_hate.mean():.1f}% of these hateful/offensive HateXplain posts as hate (out-of-domain).")
display(plaus_tab.round(4)); save_audit()
print(f"⏱ cell finished in {(_time.time()-_tc)/60:.1f} min")


In [ ]:
import time as _time; _tc = _time.time()
# ── V3-J3: Adversarial robustness on held-out UCB test hate comments ────────
import textattack, nltk
from textattack.models.wrappers import ModelWrapper
from textattack.datasets import Dataset as TADataset
from textattack import Attacker, AttackArgs
from textattack.attack_results import SuccessfulAttackResult, FailedAttackResult, SkippedAttackResult
from textattack.attack_recipes import DeepWordBugGao2018, TextFoolerJin2019
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

class V3Wrap(ModelWrapper):
    def __init__(self): self.model = xai_model
    def __call__(self, texts): return p_hate_texts(list(texts))

_samp = ucb_test[ucb_test.label == 1].sample(N_ATTACK, random_state=42)
_ta = TADataset([(str(t), 1) for t in _samp.text])
rob_rows = []
for rname, recipe in [('DeepWordBug', DeepWordBugGao2018), ('TextFooler', TextFoolerJin2019)]:
    res = Attacker(recipe.build(V3Wrap()), _ta, AttackArgs(num_examples=N_ATTACK, disable_stdout=True, random_seed=42)).attack_dataset()
    succ = [r for r in res if isinstance(r, SuccessfulAttackResult)]
    fail = sum(isinstance(r, FailedAttackResult) for r in res); skip = sum(isinstance(r, SkippedAttackResult) for r in res)
    pert = []
    for r in succ:
        try:
            o, p = r.original_result.attacked_text, r.perturbed_result.attacked_text
            pert.append(100 * len(o.all_words_diff(p)) / max(1, len(o.words)))
        except Exception: pass
    n_eff = len(succ) + fail
    rob_rows.append({'Attack': rname, 'Samples': len(res), 'Skipped (already misclassified)': skip,
                     'Clean accuracy': (len(res) - skip) / len(res), 'Successes': len(succ), 'Failures': fail,
                     'ASR': len(succ) / n_eff if n_eff else float('nan'),
                     'Robustness (1-ASR)': fail / n_eff if n_eff else float('nan'),
                     'Accuracy under attack': fail / len(res), 'Avg % words perturbed (successes)': float(np.mean(pert)) if pert else float('nan')})
    print(rob_rows[-1])
rob_tab = pd.DataFrame(rob_rows); rob_tab.to_csv(f'{XAI_DIR}/table9_adversarial.csv', index=False)
display(rob_tab.round(4))
print(f"⏱ cell finished in {(_time.time()-_tc)/60:.1f} min")


In [ ]:
import time as _time; _tc = _time.time()
# ── V3-I: Package everything for download (upload this zip back to Claude) ──
save_audit()
_zip = shutil.make_archive('/kaggle/working/v32_results', 'zip', V3_DIR,
                           base_dir='.', )
# checkpoints are large; make a second, lightweight archive without them
_light = '/kaggle/working/v32_results_light'
if os.path.exists(_light): shutil.rmtree(_light)
shutil.copytree(V3_DIR, _light, ignore=shutil.ignore_patterns('ckpt', '*.npz') )
for f in glob.glob(f'{V3_DIR}/preds/*.npz'):
    os.makedirs(f'{_light}/preds', exist_ok=True); shutil.copy(f, f'{_light}/preds/')
shutil.make_archive(_light, 'zip', _light)
print("Full archive :", _zip)
print("Light archive:", _light + '.zip', " ← upload this one (audit, runs, tables, predictions; no checkpoints)")
print(f"⏱ cell finished in {(_time.time()-_tc)/60:.1f} min")
